<a href="https://colab.research.google.com/github/rfroiman/DI_Bootcamp/blob/main/Week8/Day1/Brazilian_Ecommerce_SQL_Mini_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import zipfile

zip_path = "/content/Brazilian E-Commerce Dive Deep using SQL.zip"
extract_path = "/content/brazilian_ecommerce"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_file:
    zip_file.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [2]:
for folder, subfolders, files in os.walk("/content/brazilian_ecommerce"):
    for file in files:
        if file.endswith(".csv"):
            print(file)

product_category_name_translation.csv
olist_sellers_dataset.csv
olist_geolocation_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_order_items_dataset.csv
olist_orders_dataset.csv
olist_customers_dataset.csv


In [4]:
import pandas as pd
from pathlib import Path

dataset_folder = Path("/content/brazilian_ecommerce")

def load_csv(filename):
    matches = [
        path for path in dataset_folder.rglob(filename)
        if path.is_file()
    ]

    if not matches:
        raise FileNotFoundError(f"CSV file not found: {filename}")

    return pd.read_csv(matches[0])

df_olist_customers = load_csv("olist_customers_dataset.csv")
df_olist_sellers = load_csv("olist_sellers_dataset.csv")
df_olist_order_reviews = load_csv("olist_order_reviews_dataset.csv")
df_olist_order_items = load_csv("olist_order_items_dataset.csv")
df_olist_products = load_csv("olist_products_dataset.csv")
df_olist_geolocation = load_csv("olist_geolocation_dataset.csv")
df_product_category_name_translation = load_csv("product_category_name_translation.csv")
df_olist_orders = load_csv("olist_orders_dataset.csv")
df_olist_order_payments = load_csv("olist_order_payments_dataset.csv")

print("All 9 datasets loaded successfully!")

All 9 datasets loaded successfully!


In [5]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///brazilian_ecommerce.db")

tables = {
    "olist_customers": df_olist_customers,
    "olist_sellers": df_olist_sellers,
    "olist_order_reviews": df_olist_order_reviews,
    "olist_order_items": df_olist_order_items,
    "olist_products_dataset": df_olist_products,
    "olist_geolocation": df_olist_geolocation,
    "product_category_name_translation": df_product_category_name_translation,
    "olist_orders": df_olist_orders,
    "olist_order_payments": df_olist_order_payments
}

for table_name, dataframe in tables.items():
    dataframe.to_sql(
        table_name,
        con=engine,
        if_exists="replace",
        index=False
    )

    print(f"{table_name}: {len(dataframe)} rows")

print("SQLite database created successfully!")

olist_customers: 99441 rows
olist_sellers: 3095 rows
olist_order_reviews: 99224 rows
olist_order_items: 112650 rows
olist_products_dataset: 32951 rows
olist_geolocation: 1000163 rows
product_category_name_translation: 71 rows
olist_orders: 99441 rows
olist_order_payments: 103886 rows
SQLite database created successfully!


In [6]:
sql = """
SELECT *
FROM olist_customers
LIMIT 5;
"""

df_sql = pd.read_sql_query(sql, con=engine)

display(df_sql)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## Query 1: Count and Percentage of Orders Purchased in January 2018 with 5 Review Score

In [7]:
sql = """
SELECT
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT CASE
        WHEN r.review_score = 5 THEN o.order_id
    END) AS five_star_orders,
    ROUND(
        COUNT(DISTINCT CASE
            WHEN r.review_score = 5 THEN o.order_id
        END) * 100.0 / COUNT(DISTINCT o.order_id),
        2
    ) AS five_star_percentage
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS r
    ON o.order_id = r.order_id
WHERE o.order_purchase_timestamp >= '2018-01-01'
    AND o.order_purchase_timestamp < '2018-02-01';
"""

df_sql = pd.read_sql_query(sql, con=engine)

display(df_sql)

,total_orders,five_star_orders,five_star_percentage
0,7269,4077,56.09


### Query 1 — Analysis

In January 2018, a total of **7,269 orders** were placed on the Brazilian e-commerce platform. Of these, **4,077 orders (56.09%)** received a 5-star review.

This indicates that more than half of the orders placed during this period received the highest possible customer rating. However, this percentage includes orders without reviews in the total, so it should not be interpreted as the percentage of 5-star ratings among reviewed orders only.

## Query 2: Customer Purchase Trend Year-on-Year


In [8]:
sql = """
SELECT
    strftime('%Y', order_purchase_timestamp) AS year,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_id) AS total_customers
FROM olist_orders
WHERE order_purchase_timestamp IS NOT NULL
GROUP BY strftime('%Y', order_purchase_timestamp)
ORDER BY year;
"""

df_sql = pd.read_sql_query(sql, con=engine)

display(df_sql)

,year,total_orders,total_customers
0,2016,329,329
1,2017,45101,45101
2,2018,54011,54011


### Query 2 — Analysis

The dataset shows **329 orders in 2016**, **45,101 orders in 2017**, and **54,011 orders in 2018**.

Orders increased significantly between 2016 and 2017, followed by an additional increase of approximately **19.8%** in 2018.

However, the dataset does not cover complete calendar years for all periods. Therefore, these figures represent the observed purchase trend rather than directly comparable full-year growth rates.

Additionally, the customer counts in this query refer to order-level customer IDs, not necessarily unique individuals.

## Query 3: Average Order Values of Customers

In [9]:
sql = """
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(p.payment_value) AS order_value
    FROM olist_orders AS o
    INNER JOIN olist_customers AS c
        ON o.customer_id = c.customer_id
    INNER JOIN olist_order_payments AS p
        ON o.order_id = p.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
)
SELECT
    customer_unique_id,
    COUNT(order_id) AS total_orders,
    ROUND(AVG(order_value), 2) AS average_order_value
FROM order_totals
GROUP BY customer_unique_id
ORDER BY average_order_value DESC;
"""

df_sql = pd.read_sql_query(sql, con=engine)

display(df_sql.head(10))

,customer_unique_id,total_orders,average_order_value
0,0a0a92112bd4c708ca5fde585afaa872,1,13664.08
1,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88
2,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31
3,459bef486812aa25204be022145caa62,1,6922.21
4,ff4159b92c40ebe40454e3e6a7c35ed6,1,6726.66
5,4007669dec559734d6f53e029e360987,1,6081.54
6,5d0a2980b292d049061542014e8960bf,1,4809.44
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.34
8,48e1ac109decbb87765a3eade6854098,1,4681.78
9,edde2314c6c30e864a128ac95d6b2112,1,4513.32


### Query 3 — Analysis

The query calculates the average order value for each unique customer based on their payment history.

The highest average order value was **BRL 13,664.08**, followed by **BRL 7,274.88** and **BRL 6,929.31**.

All top 10 customers placed only one order, meaning their average order value equals their individual purchase amount.

These results highlight the customers with the highest average spending per order, rather than the customers with the highest total spending or purchase frequency.

## Query 4: Top 5 Cities with Highest Revenue (2016–2018)

In [10]:
sql = """
WITH order_payments AS (
    SELECT
        order_id,
        SUM(payment_value) AS total_payment
    FROM olist_order_payments
    GROUP BY order_id
)
SELECT
    c.customer_city AS city,
    c.customer_state AS state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(p.total_payment), 2) AS total_revenue
FROM olist_orders AS o
INNER JOIN olist_customers AS c
    ON o.customer_id = c.customer_id
INNER JOIN order_payments AS p
    ON o.order_id = p.order_id
WHERE o.order_purchase_timestamp >= '2016-01-01'
    AND o.order_purchase_timestamp < '2019-01-01'
GROUP BY
    c.customer_city,
    c.customer_state
ORDER BY total_revenue DESC
LIMIT 5;
"""

df_sql = pd.read_sql_query(sql, con=engine)

display(df_sql)

,city,state,total_orders,total_revenue
0,sao paulo,SP,15540,2203373.09
1,rio de janeiro,RJ,6882,1161927.36
2,belo horizonte,MG,2773,421765.12
3,brasilia,DF,2131,354216.78
4,curitiba,PR,1521,247392.48


### Query 4 — Analysis

São Paulo generated the highest revenue between 2016 and 2018, with **BRL 2,203,373.09** from 15,540 orders.

Rio de Janeiro ranked second, generating **BRL 1,161,927.36**, followed by Belo Horizonte, Brasília, and Curitiba.

São Paulo and Rio de Janeiro together accounted for approximately **76.7% of the revenue generated by these five cities**.

These results demonstrate the strong concentration of e-commerce sales in major Brazilian urban centers.

## Query 5: State-wise Revenue (2016–2018)

In [11]:
sql = """
WITH order_payments AS (
    SELECT
        order_id,
        SUM(payment_value) AS total_payment
    FROM olist_order_payments
    GROUP BY order_id
)
SELECT
    c.customer_state AS state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(p.total_payment), 2) AS total_revenue,
    ROUND(AVG(p.total_payment), 2) AS average_order_value
FROM olist_orders AS o
INNER JOIN olist_customers AS c
    ON o.customer_id = c.customer_id
INNER JOIN order_payments AS p
    ON o.order_id = p.order_id
WHERE o.order_purchase_timestamp >= '2016-01-01'
    AND o.order_purchase_timestamp < '2019-01-01'
GROUP BY c.customer_state
ORDER BY total_revenue DESC;
"""

df_sql = pd.read_sql_query(sql, con=engine)

display(df_sql)

,state,total_orders,total_revenue,average_order_value
0,SP,41745,5998226.96,143.69
1,RJ,12852,2144379.69,166.85
2,MG,11635,1872257.26,160.92
3,RS,5466,890898.54,162.99
4,PR,5045,811156.38,160.78
5,SC,3637,623086.43,171.32
6,BA,3380,616645.82,182.44
7,DF,2140,355141.08,165.95
8,GO,2020,350092.31,173.31
9,ES,2033,325967.55,160.34


### Query 5 — Analysis

São Paulo generated the highest revenue, with **BRL 5,998,226.96** from 41,745 orders, followed by Rio de Janeiro (BRL 2,144,379.69) and Minas Gerais (BRL 1,872,257.26).

The results show that the Southeast region dominates e-commerce revenue in this dataset.

However, states with lower sales volumes sometimes have higher average order values. For example, Paraíba recorded the highest average order value at **BRL 264.08**, compared with **BRL 143.69** in São Paulo.

This suggests that revenue performance is influenced by both order volume and average spending per order.

## Query 6: Top Sellers by Goods Sold, Revenue, Customer Count and 5-Star Reviews

In [12]:
sql = """
WITH seller_sales AS (
    SELECT
        i.seller_id,
        COUNT(*) AS goods_sold,
        ROUND(SUM(i.price), 2) AS total_revenue,
        COUNT(DISTINCT c.customer_unique_id) AS total_customers
    FROM olist_order_items AS i
    INNER JOIN olist_orders AS o
        ON i.order_id = o.order_id
    INNER JOIN olist_customers AS c
        ON o.customer_id = c.customer_id
    GROUP BY i.seller_id
),
seller_reviews AS (
    SELECT
        seller_id,
        COUNT(*) AS five_star_reviews
    FROM (
        SELECT DISTINCT
            i.seller_id,
            i.order_id
        FROM olist_order_items AS i
        INNER JOIN olist_order_reviews AS r
            ON i.order_id = r.order_id
        WHERE r.review_score = 5
    )
    GROUP BY seller_id
)
SELECT
    s.seller_id,
    s.goods_sold,
    s.total_revenue,
    s.total_customers,
    COALESCE(r.five_star_reviews, 0) AS five_star_reviews
FROM seller_sales AS s
LEFT JOIN seller_reviews AS r
    ON s.seller_id = r.seller_id
ORDER BY
    s.goods_sold DESC,
    s.total_revenue DESC
LIMIT 10;
"""

df_sql = pd.read_sql_query(sql, con=engine)

display(df_sql)

,seller_id,goods_sold,total_revenue,total_customers,five_star_reviews
0,6560211a19b47992c3666cc44a7e94c0,2033,123304.83,1824,945
1,4a3ca9315b744ce9f8e9374361493884,1987,200472.92,1792,865
2,1f50f920176fa81dab994f9023523100,1931,106939.21,1388,841
3,cc419e0650a3c5ba77189a1882b7556a,1775,104288.42,1657,996
4,da8622b14eb17ae2831f4ac5b9dab84a,1551,160236.57,1275,771
5,955fee9216a65b617aa5c0531780ce60,1499,135171.70,1282,719
6,1025f0e2d44d7041d6cf58b6550e0bfa,1428,138968.55,899,502
7,7c67e1448b00f6e969d365cea6b010ab,1364,187923.89,971,343
8,ea8482cd71df3c1969d7b9473ff13abc,1203,37177.52,1133,601
9,7a67c85e85bb2ce8582c35f2203ad736,1171,141745.53,1155,710


### Query 6 — Analysis

The seller identified as `6560211a...` ranked first in sales volume, with **2,033 items sold** and **1,824 unique customers**.

However, the seller `4a3ca931...` generated the highest revenue among the top 10 sellers by volume, reaching **BRL 200,472.92**.

The seller `cc419e06...` had the highest number of orders associated with 5-star reviews within this group, totaling **996**.

These findings demonstrate that sales volume, revenue, customer reach, and customer satisfaction are different dimensions of seller performance.

Since reviews are recorded at the order level, a review associated with multiple sellers should not be interpreted as an independent evaluation of each seller.

## Query 7: Delivery Success Rate Across States

In [13]:
sql = """
SELECT
    c.customer_state AS state,
    COUNT(o.order_id) AS total_orders,
    SUM(
        CASE
            WHEN o.order_status = 'delivered' THEN 1
            ELSE 0
        END
    ) AS delivered_orders,
    SUM(
        CASE
            WHEN o.order_status <> 'delivered' THEN 1
            ELSE 0
        END
    ) AS not_delivered_orders,
    ROUND(
        SUM(
            CASE
                WHEN o.order_status = 'delivered' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(o.order_id),
        2
    ) AS delivery_success_rate
FROM olist_orders AS o
INNER JOIN olist_customers AS c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY delivery_success_rate DESC;
"""

df_sql = pd.read_sql_query(sql, con=engine)

display(df_sql)

,state,total_orders,delivered_orders,not_delivered_orders,delivery_success_rate
0,AC,81,80,1,98.77
1,AP,68,67,1,98.53
2,ES,2033,1995,38,98.13
3,MS,715,701,14,98.04
4,AM,148,145,3,97.97
5,TO,280,274,6,97.86
6,RS,5466,5345,121,97.79
7,RN,485,474,11,97.73
8,MT,907,886,21,97.68
9,PR,5045,4923,122,97.58


### Query 7 — Analysis

Acre (AC) recorded the highest delivery success rate at **98.77%**, followed by Amapá (AP) at **98.53%** and Espírito Santo (ES) at **98.13%**.

São Paulo (SP), the state with the largest order volume, achieved a delivery success rate of **97.02%**, with 40,501 delivered orders.

Roraima (RR) recorded the lowest rate at **89.13%**, although its sample included only 46 orders.

Overall, 26 of the 27 Brazilian states and federal districts achieved delivery success rates above 95%.

This metric measures the proportion of orders marked as delivered in the dataset. Orders with other statuses are not necessarily failed deliveries, and smaller samples may produce less stable percentages.

## Query 8: Preferred Payment Form for Different Product Categories

In [14]:
sql = """
WITH category_orders AS (
    SELECT DISTINCT
        p.product_category_name,
        i.order_id
    FROM olist_order_items AS i
    INNER JOIN olist_products_dataset AS p
        ON i.product_id = p.product_id
    WHERE p.product_category_name IS NOT NULL
),
payment_counts AS (
    SELECT
        COALESCE(
            t.product_category_name_english,
            co.product_category_name
        ) AS category,
        pay.payment_type,
        COUNT(*) AS payment_count
    FROM category_orders AS co
    INNER JOIN olist_order_payments AS pay
        ON co.order_id = pay.order_id
    LEFT JOIN product_category_name_translation AS t
        ON co.product_category_name = t.product_category_name
    GROUP BY
        category,
        pay.payment_type
),
ranked_payments AS (
    SELECT
        category,
        payment_type,
        payment_count,
        ROW_NUMBER() OVER (
            PARTITION BY category
            ORDER BY payment_count DESC, payment_type
        ) AS payment_rank
    FROM payment_counts
)
SELECT
    category,
    payment_type AS preferred_payment_method,
    payment_count
FROM ranked_payments
WHERE payment_rank = 1
ORDER BY payment_count DESC;
"""

df_sql = pd.read_sql_query(sql, con=engine)

display(df_sql)

,category,preferred_payment_method,payment_count
0,bed_bath_table,credit_card,7565
1,health_beauty,credit_card,6898
2,sports_leisure,credit_card,5923
3,furniture_decor,credit_card,4933
4,computers_accessories,credit_card,4745
...,...,...,...
68,cds_dvds_musicals,credit_card,9
69,portateis_cozinha_e_preparadores_de_alimentos,credit_card,8
70,fashion_childrens_clothes,credit_card,5
71,pc_gamer,credit_card,5


### Query 8 — Analysis

The query analyzed **73 product categories** to identify the most frequently used payment method in each category.

**Credit card** was the preferred payment method in almost all categories, including **bed_bath_table** (7,565 payment records), **health_beauty** (6,898), and **sports_leisure** (5,923).

The **security_and_services** category was an exception, with **boleto** as its preferred payment method, based on only one payment record.

These findings indicate that credit cards dominate payment preferences across different product categories in the Brazilian e-commerce dataset.

However, payment counts represent payment records associated with each category, rather than necessarily unique orders or customers.

## Query 9: Distance Between Cities

In [15]:
import math
import sqlite3

# Create a connection to the SQLite database
connection = sqlite3.connect("brazilian_ecommerce.db")

# Register mathematical functions in SQLite
connection.create_function("RADIANS", 1, math.radians)
connection.create_function("SIN", 1, math.sin)
connection.create_function("COS", 1, math.cos)
connection.create_function("ASIN", 1, math.asin)
connection.create_function("SQRT", 1, math.sqrt)

sql = """
WITH city_coordinates AS (
    SELECT
        geolocation_city AS city,
        geolocation_state AS state,
        AVG(geolocation_lat) AS latitude,
        AVG(geolocation_lng) AS longitude
    FROM olist_geolocation
    WHERE (
        geolocation_city = 'sao paulo' AND geolocation_state = 'SP'
    ) OR (
        geolocation_city = 'rio de janeiro' AND geolocation_state = 'RJ'
    ) OR (
        geolocation_city = 'belo horizonte' AND geolocation_state = 'MG'
    ) OR (
        geolocation_city = 'brasilia' AND geolocation_state = 'DF'
    ) OR (
        geolocation_city = 'curitiba' AND geolocation_state = 'PR'
    )
    GROUP BY geolocation_city, geolocation_state
)
SELECT
    a.city AS origin_city,
    b.city AS destination_city,
    ROUND(
        6371 * 2 * ASIN(
            MIN(
                1.0,
                SQRT(
                    (1 - COS(RADIANS(b.latitude - a.latitude))) / 2
                    +
                    COS(RADIANS(a.latitude))
                    * COS(RADIANS(b.latitude))
                    * (1 - COS(RADIANS(b.longitude - a.longitude))) / 2
                )
            )
        ),
        2
    ) AS distance_km
FROM city_coordinates AS a
CROSS JOIN city_coordinates AS b
WHERE a.city = 'sao paulo'
    AND a.state = 'SP'
    AND NOT (a.city = b.city AND a.state = b.state)
ORDER BY distance_km ASC;
"""

df_sql = pd.read_sql_query(sql, con=connection)

display(df_sql)

connection.close()

,origin_city,destination_city,distance_km
0,sao paulo,curitiba,339.45
1,sao paulo,rio de janeiro,346.99
2,sao paulo,belo horizonte,492.09
3,sao paulo,brasilia,874.13


### Query 9 — Analysis

The query calculated the approximate geographical distances between São Paulo and four major Brazilian cities using the Haversine formula.

**Curitiba** was the closest city to São Paulo, at approximately **339.45 km**, followed by **Rio de Janeiro (346.99 km)** and **Belo Horizonte (492.09 km)**.

**Brasília** was the farthest city among those analyzed, at approximately **874.13 km** from São Paulo.

These results demonstrate how geographical coordinates can be used in SQL to calculate distances between cities.

However, the calculations represent straight-line distances based on average geographical coordinates from the dataset, rather than actual driving distances.

This type of analysis can support logistics planning, delivery optimization, and regional distribution studies.